# 03 — Modelo 2: filtragem colaborativa baseada em usuários

A ideia: **quem gostou dos mesmos animes que você provavelmente gosta de outros que você ainda não viu.**

1. o usuário novo escolhe até 3 personas, e isso vira um vetor de notas (nota 10 nos animes escolhidos);
2. buscamos os usuários mais parecidos com esse vetor (`NearestNeighbors` com `algorithm="brute"`);
3. recomendamos os animes com a maior soma de notas entre esses vizinhos.

O que muda entre os modelos é como medimos "parecido": distância **Cosseno**, **Manhattan** ou **Euclidiana**.

In [1]:
import numpy as np
import pandas as pd

from joblib import load

from src.config import ANIMES_CLUSTERS, K_VIZINHOS, MATRIZ_USUARIO_ANIME, METRICAS_DISTANCIA
from src.graficos import ROTULOS
from src.models import (
    ajustar_vizinhos,
    montar_vetores,
    recomendar_por_itens,
    recomendar_por_usuarios,
    similaridade_entre_animes,
)

animes = pd.read_parquet(ANIMES_CLUSTERS)
matriz = load(MATRIZ_USUARIO_ANIME)
print(matriz.shape)

(20000, 6166)


## 1. O usuário novo

Como exemplo, alguém que escolheu três personas:

In [2]:
escolhidos = ["Death Note", "Fullmetal Alchemist: Brotherhood", "Code Geass: Hangyaku no Lelouch"]
indices = animes.index[animes["Name"].isin(escolhidos)].tolist()

vetor = montar_vetores([indices], n_animes=matriz.shape[1])
animes.loc[indices, ["Name", "Cluster", "Descrição do cluster"]]

,Name,Cluster,Descrição do cluster
1174,Death Note,16,R - 17+ (violence & profanity) · TV · 2-13 epi...
1203,Code Geass: Hangyaku no Lelouch,5,Mecha · Sci-Fi · Original · Action
2234,Fullmetal Alchemist: Brotherhood,6,27+ episódios · TV · Até 2000 · Shounen


`vetor` tem uma linha (o usuário novo) e uma coluna por anime: 10 nos três escolhidos e 0 em todo o resto.

## 2. Recomendações com cada distância

In [3]:
modelos = {metrica: ajustar_vizinhos(matriz, metrica) for metrica in METRICAS_DISTANCIA}

recomendacoes = pd.DataFrame(
    {
        ROTULOS[metrica]: animes["Name"].to_numpy()[recomendar_por_usuarios(knn, matriz, vetor)[0]]
        for metrica, knn in modelos.items()
    }
)
recomendacoes.index += 1
recomendacoes

,Cosine,Manhattan,Euclidean
1,Code Geass: Hangyaku no Lelouch R2,Shingeki no Kyojin,Shingeki no Kyojin
2,Shingeki no Kyojin,Naruto,Cowboy Bebop
3,Naruto,One Punch Man,One Punch Man
4,Fullmetal Alchemist,Code Geass: Hangyaku no Lelouch R2,Naruto
5,Sword Art Online,Tengen Toppa Gurren Lagann,Neon Genesis Evangelion
6,Steins;Gate,Steins;Gate,Tengen Toppa Gurren Lagann
7,Ao no Exorcist,Cowboy Bebop,Steins;Gate
8,Hunter x Hunter (2011),Elfen Lied,Angel Beats!
9,Mirai Nikki,Sword Art Online,Sword Art Online
10,Elfen Lied,Boku no Hero Academia,Boku no Hero Academia


## 3. Por que as distâncias recomendam coisas diferentes?

Olhando quem são os vizinhos que cada distância encontra:

In [4]:
avaliacoes_por_usuario = matriz.getnnz(axis=1)
notas_nos_escolhidos = matriz[:, indices].toarray()

vizinhos = {
    ROTULOS[metrica]: knn.kneighbors(vetor, n_neighbors=K_VIZINHOS, return_distance=False)[0]
    for metrica, knn in modelos.items()
}
pd.DataFrame(
    {
        "Animes avaliados por vizinho (mediana)": {m: np.median(avaliacoes_por_usuario[v]) for m, v in vizinhos.items()},
        "Escolhidos que o vizinho avaliou (média, de 3)": {m: (notas_nos_escolhidos[v] > 0).sum(axis=1).mean() for m, v in vizinhos.items()},
        "Nota média do vizinho nos escolhidos": {m: notas_nos_escolhidos[v][notas_nos_escolhidos[v] > 0].mean() for m, v in vizinhos.items()},
    }
).round(1)

,Animes avaliados por vizinho (mediana),"Escolhidos que o vizinho avaliou (média, de 3)",Nota média do vizinho nos escolhidos
Cosine,54.0,3.0,9.4
Manhattan,52.0,1.8,7.7
Euclidean,55.0,1.8,7.1


- **Cosseno** olha a *direção* do vetor, não o tamanho. Os vizinhos são usuários que avaliaram os três animes escolhidos e deram notas altas (média 9,4), ou seja, gente com o mesmo gosto.
- **Manhattan** e **Euclidiana** somam diferenças em *todas* as colunas. O usuário novo tem zero em quase tudo, então cada anime a mais que o outro usuário avaliou aumenta a distância.
  Os vizinhos acabam sendo os usuários com menos avaliações da base (perto do mínimo de 50) e que viram, em média, menos de 2 dos 3 escolhidos.

Por isso Manhattan e Euclidiana tendem a recomendar o que usuários com poucos animes assistiram: os títulos mais conhecidos.

## 4. Aprimoramento: baseline baseado em itens

Uma alternativa clássica é comparar **animes** em vez de usuários: dois animes são parecidos quando as mesmas pessoas dão notas parecidas para eles.
Não faz parte da monografia; entra aqui como ponto de comparação para a avaliação do próximo notebook.

In [5]:
similaridade = similaridade_entre_animes(matriz)

pd.Series(
    animes["Name"].to_numpy()[recomendar_por_itens(similaridade, vetor)[0]],
    index=range(1, 11),
    name=ROTULOS["item"],
).to_frame()

,Item-based
1,Code Geass: Hangyaku no Lelouch R2
2,Shingeki no Kyojin
3,Steins;Gate
4,Sword Art Online
5,One Punch Man
6,Angel Beats!
7,Toradora!
8,Tokyo Ghoul
9,Naruto
10,No Game No Life
